# 🌟 WP6 — PrometheusStar Strategy Visualisation Dashboard

**Prometheus v0.97 · Workplan 6 · Strategy Visualisation Tool**

This notebook demonstrates all six dashboard pages as **static matplotlib figures**,
making them fully runnable in Google Colab without a Streamlit server.

| Section | Dashboard page |
|---------|---------------|
| §1 Setup | — |
| §2 Overview | Overview |
| §3 Learning Curves | Learning Curves |
| §4 Strategy Evolution | Strategy Evolution |
| §5 Agent Inspector | Agent Inspector |
| §6 OOD Dashboard | OOD Dashboard |
| §7 Value Learning | Value Learning |

To run the **live Streamlit dashboard** locally:
```bash
pip install streamlit pandas matplotlib
streamlit run prometheus_dashboard.py
```

## §1 — Setup

In [ ]:
# Install / upgrade dependencies (Colab)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'matplotlib', 'numpy'], check=True)
print('Dependencies ready.')

In [ ]:
import sys, os

# If running in Colab, clone the repo
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/pmineiro/Prometheus_v0_PoC.git'], check=True)
    os.chdir('Prometheus_v0_PoC')

# Add repo root to path
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print('Working directory:', repo_root)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from IPython.display import display

from prometheus.strategy_data import (
    generate_synthetic_openra_run,
    generate_synthetic_microrts_run,
    flatten_generation_log,
    agent_params_over_generations,
    stage_summary_df,
    ood_summary_df,
    value_weight_df,
)

print('Imports OK.')

In [ ]:
# ── Shared constants ──────────────────────────────────────────────────────────
STAGE_COLOURS = [
    '#4C72B0', '#DD8452', '#55A868', '#C44E52',
    '#8172B3', '#937860', '#DA8BC3', '#8C8C8C',
]

# Generate synthetic run data
openra_results   = generate_synthetic_openra_run(seed=42, n_stages=4)
microrts_results = generate_synthetic_microrts_run(seed=7,  n_stages=3)

print(f'OpenRA:   {len(openra_results)} stages, '
      f'{sum(r["generations_run"] for r in openra_results)} total generations')
print(f'MicroRTS: {len(microrts_results)} stages, '
      f'{sum(r["generations_run"] for r in microrts_results)} total generations')

---
## §2 — Overview Page

High-level summary: stage badges (target met / missed), key metrics, and the stage summary table.

In [ ]:
summary = stage_summary_df(openra_results)

n_met   = int(summary['target_met'].sum())
n_total = len(summary)
total_g = int(summary['generations_run'].sum())
mean_wr = float(summary['best_win_rate'].mean())

print('=' * 50)
print(f'  Stages run:          {n_total}')
print(f'  Targets met:         {n_met} / {n_total}')
print(f'  Total generations:   {total_g}')
print(f'  Mean best win rate:  {mean_wr:.1%}')
print('=' * 50)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
ax.axis('off')

for i, r in enumerate(openra_results):
    met    = r['best_win_rate'] >= r['target']
    colour = '#2ecc71' if met else '#e74c3c'
    icon   = '✅' if met else '❌'
    x      = i / len(openra_results)
    w      = 0.9 / len(openra_results)
    rect   = plt.Rectangle((x, 0), w, 1, color=colour, alpha=0.85, transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(x + w/2, 0.70, f"Stage {r['stage']}",
            transform=ax.transAxes, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(x + w/2, 0.45, r['name'],
            transform=ax.transAxes, ha='center', va='center',
            fontsize=8, color='white')
    ax.text(x + w/2, 0.20,
            f"{icon} {r['best_win_rate']:.1%} / {r['target']:.1%}",
            transform=ax.transAxes, ha='center', va='center',
            fontsize=8, color='white')

ax.set_title('Stage Badges — OpenRA Curriculum Run', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
display_cols = ['stage','name','difficulty','target','best_win_rate',
                'target_met','generations_run','wall_clock_s']
display_cols = [c for c in display_cols if c in summary.columns]
display(summary[display_cols].style.format({
    'target':        '{:.1%}',
    'best_win_rate': '{:.1%}',
    'wall_clock_s':  '{:.1f}s',
}))

---
## §3 — Learning Curves Page

Win-rate trajectories across all curriculum stages (cumulative + per-stage zoom).

In [ ]:
df = flatten_generation_log(openra_results)

fig, ax = plt.subplots(figsize=(11, 4))

for i, r in enumerate(openra_results):
    sub   = df[df['stage'] == r['stage']]
    c     = STAGE_COLOURS[i % len(STAGE_COLOURS)]
    label = f"S{r['stage']} {r['name']}"
    ax.plot(sub['cumulative_generation'], sub['best_win_rate'],
            color=c, lw=2, label=f'{label} (best)')
    ax.plot(sub['cumulative_generation'], sub['mean_win_rate'],
            color=c, lw=1, linestyle='--', alpha=0.6, label=f'{label} (mean)')
    ax.axhline(r['target'], color=c, lw=0.8, linestyle=':', alpha=0.5)

ax.set_xlabel('Cumulative Generation', fontsize=11)
ax.set_ylabel('Win Rate',              fontsize=11)
ax.set_title('Win Rate over Generations — OpenRA Curriculum', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Per-stage zoom (all stages in a 2×2 grid)
n = len(openra_results)
ncols = 2
nrows = (n + 1) // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 3.5), squeeze=False)

for idx, r in enumerate(openra_results):
    ax  = axes[idx // ncols][idx % ncols]
    sub = df[df['stage'] == r['stage']]
    ax.plot(sub['generation'], sub['best_win_rate'], lw=2, label='Best')
    ax.plot(sub['generation'], sub['mean_win_rate'], lw=1, linestyle='--',
            alpha=0.7, label='Mean')
    ax.axhline(r['target'], color='red', lw=1, linestyle=':', label=f"Target {r['target']:.0%}")
    ax.set_title(f"Stage {r['stage']}: {r['name']} ({r.get('difficulty','')})",
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Generation')
    ax.set_ylabel('Win Rate')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# Hide unused axes
for idx in range(n, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle('Per-Stage Win Rate Zoom', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## §4 — Strategy Evolution Page

Parallel coordinates (stage-end parameters), per-parameter time series, and a radar (spider) chart.

In [ ]:
summary   = stage_summary_df(openra_results)
param_cols = [c for c in summary.columns if c.startswith('param_')]

fig, ax = plt.subplots(figsize=(10, 4))
x_pos   = list(range(len(param_cols)))
cmap    = cm.get_cmap('tab10')

for i, (_, row) in enumerate(summary.iterrows()):
    vals   = [float(row[c]) for c in param_cols]
    colour = cmap(i / max(1, len(summary) - 1))
    ax.plot(x_pos, vals, marker='o', color=colour, lw=2,
            label=f"S{int(row['stage'])} {row['name']}")

ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('param_', '') for c in param_cols], rotation=25, ha='right')
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel('Parameter value')
ax.set_title('Parallel Coordinates — Strategy Parameters at End of Each Stage',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
params_df = agent_params_over_generations(openra_results)
all_params = sorted(params_df['param_name'].unique())

fig, axes = plt.subplots(len(all_params), 1,
                         figsize=(11, 2.5 * len(all_params)), sharex=True)

for ax, param in zip(axes, all_params):
    sub = params_df[params_df['param_name'] == param]
    for i, stage in enumerate(sorted(sub['stage'].unique())):
        ss = sub[sub['stage'] == stage]
        r  = openra_results[stage - 1]
        ax.plot(ss['cumulative_generation'], ss['param_value'],
                color=STAGE_COLOURS[i % len(STAGE_COLOURS)], lw=2,
                label=f"S{stage} {r['name']}")
    ax.set_ylabel(param, fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc='upper left', ncol=2)

axes[-1].set_xlabel('Cumulative generation')
fig.suptitle('Strategy Parameter Evolution over Generations',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
labels = [c.replace('param_', '') for c in param_cols]
N      = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'polar': True})
cmap    = cm.get_cmap('tab10')

for i, (_, row) in enumerate(summary.iterrows()):
    vals   = [float(row[c]) for c in param_cols] + [float(row[param_cols[0]])]
    colour = cmap(i / max(1, len(summary) - 1))
    ax.plot(angles, vals, color=colour, lw=2,
            label=f"S{int(row['stage'])} {row['name']}")
    ax.fill(angles, vals, color=colour, alpha=0.07)

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Strategy Radar Chart\n(stage-end parameters)',
             y=1.12, fontsize=12, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.2), fontsize=8)
plt.tight_layout()
plt.show()

---
## §5 — Agent Inspector Page

Per-generation parameter snapshot for a chosen stage and generation.

In [ ]:
# Inspect Stage 3 (Tactical Response), last generation
INSPECT_STAGE = 3

r       = openra_results[INSPECT_STAGE - 1]
gen_log = r['generation_log']
g_data  = gen_log[-1]   # last generation
params  = g_data.get('agent_params') or r.get('agent_params', {})

print(f"Stage {r['stage']}: {r['name']}  ({r.get('difficulty','')}")
print(f"Generation {g_data['generation']}  "
      f"best_wr={g_data['best_win_rate']:.2%}  "
      f"mean_wr={g_data.get('mean_win_rate', 0):.2%}  "
      f"target={r['target']:.2%} → "
      f"{'MET' if g_data['best_win_rate'] >= r['target'] else 'NOT MET'}")

pnames = list(params.keys())
pvals  = [float(params[k]) for k in pnames]

fig, ax = plt.subplots(figsize=(8, 3))
bars    = ax.barh(pnames, pvals, color=STAGE_COLOURS[:len(pnames)])
ax.set_xlim(0, 1)
ax.set_xlabel('Parameter value')
ax.set_title(f"Stage {INSPECT_STAGE} · Gen {g_data['generation']} — Agent Parameters",
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, pvals):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
log_df = pd.DataFrame([
    {'gen': g['generation'],
     'best_win_rate': g['best_win_rate'],
     'mean_win_rate': g.get('mean_win_rate', None)}
    for g in gen_log
])
display(log_df.style.format({'best_win_rate': '{:.2%}', 'mean_win_rate': '{:.2%}'})
               .set_caption(f'Stage {INSPECT_STAGE} — generation log'))

---
## §6 — OOD Dashboard Page

Out-of-distribution detection benchmark results (WP2 integration).

In [ ]:
try:
    from benchmarks.ood_benchmark import OODBenchmark
    bench   = OODBenchmark(n_samples=200, seed=42)
    results = bench.run_all()
    print(f'OOD benchmark ran {len(results)} detectors.')
    ood_available = True
except Exception as exc:
    print(f'OOD benchmark unavailable ({exc}); using synthetic data.')
    ood_available = False
    results = [
        {'detector': 'IsolationForest', 'auroc': 0.91, 'tpr': 0.83, 'fpr': 0.09,
         'in_dist_flagged': 0.05, 'mild_flagged': 0.62, 'severe_flagged': 0.94},
        {'detector': 'MahalanobisDistance','auroc': 0.88, 'tpr': 0.80, 'fpr': 0.12,
         'in_dist_flagged': 0.07, 'mild_flagged': 0.58, 'severe_flagged': 0.91},
        {'detector': 'EnsembleOOD',      'auroc': 0.94, 'tpr': 0.87, 'fpr': 0.07,
         'in_dist_flagged': 0.04, 'mild_flagged': 0.68, 'severe_flagged': 0.96},
    ]

In [ ]:
df_ood = ood_summary_df(results)
display(df_ood.style.format({'auroc': '{:.3f}', 'tpr': '{:.3f}', 'fpr': '{:.3f}'}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
det_labels = df_ood['detector'] if 'detector' in df_ood.columns else df_ood.index

for ax, metric in zip(axes, ['auroc', 'tpr', 'fpr']):
    if metric in df_ood.columns:
        ax.barh(det_labels, df_ood[metric], color='#4C72B0')
        ax.set_xlabel(metric.upper())
        ax.set_title(metric.upper(), fontsize=12, fontweight='bold')
        ax.set_xlim(0, 1)
        ax.grid(axis='x', alpha=0.3)

fig.suptitle('OOD Detection Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
flag_cols = [c for c in df_ood.columns if c.endswith('_flagged')]
if flag_cols:
    fig, ax = plt.subplots(figsize=(9, 3.5))
    x     = np.arange(len(df_ood))
    width = 0.25
    for j, col in enumerate(flag_cols):
        ax.bar(x + j * width, df_ood[col], width,
               label=col.replace('_flagged','').replace('_',' '))
    ax.set_xticks(x + width)
    ax.set_xticklabels(det_labels, rotation=20, ha='right')
    ax.set_ylabel('Flagging rate')
    ax.set_ylim(0, 1.05)
    ax.set_title('Scenario Flagging Rates', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No flagging-rate columns available.')

---
## §7 — Value Learning Page

Bradley-Terry IRL weight convergence demonstration (WP4 integration).

In [ ]:
N_FEATS  = 5
N_EPOCHS = 80

try:
    from prometheus.value_learning import ValueLearningAgent, SyntheticOracle
    oracle  = SyntheticOracle(n_features=N_FEATS, seed=42)
    agent   = ValueLearningAgent(n_features=N_FEATS)
    history, true_w = agent.train_to_convergence(oracle, n_epochs=N_EPOCHS)
    print(f'Value learning converged in {len(history)} epochs.')
    vl_ok = True
except Exception as exc:
    print(f'Value learning module unavailable ({exc}); using synthetic history.')
    vl_ok   = False
    true_w  = np.array([0.6, -0.3, 0.8, -0.1, 0.4])
    rng     = np.random.default_rng(42)
    history = []
    w       = rng.normal(0, 0.5, N_FEATS)
    for ep in range(N_EPOCHS):
        w = w + 0.04 * (true_w - w) + rng.normal(0, 0.02, N_FEATS)
        history.append(w.copy())

In [ ]:
feat_names = [f'φ[{i}]' for i in range(N_FEATS)]
vw_df      = value_weight_df(history, feat_names)

fig, ax = plt.subplots(figsize=(11, 4))
for name in feat_names:
    sub = vw_df[vw_df['feature_name'] == name]
    ax.plot(sub['epoch'], sub['weight_value'], lw=1.8, label=name)

if true_w is not None:
    for j, tw in enumerate(true_w):
        ax.axhline(tw, color='grey', lw=0.6, linestyle='--', alpha=0.4)

ax.set_xlabel('Epoch')
ax.set_ylabel('Weight value')
ax.set_title('IRL Weight Convergence\n(dashed lines = true weights)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
if true_w is not None:
    final_w = np.array(history[-1])
    x       = np.arange(N_FEATS)

    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.bar(x - 0.2, final_w, 0.4, label='Learned', color='#4C72B0')
    ax.bar(x + 0.2, true_w,  0.4, label='True',    color='#DD8452', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(feat_names)
    ax.set_ylabel('Weight')
    ax.set_title('Learned vs. True Reward Weights',
                 fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    cos_sim = float(np.dot(final_w, true_w) / (np.linalg.norm(final_w) * np.linalg.norm(true_w) + 1e-9))
    mse     = float(np.mean((final_w - true_w)**2))
    print(f'Cosine similarity learned ↔ true: {cos_sim:.4f}')
    print(f'MSE:                               {mse:.6f}')

---
## Summary

This notebook demonstrated all six pages of the **WP6 PrometheusStar Strategy
Visualisation Dashboard** as static figures:

| Page | Key chart(s) |
|------|--------------|
| **Overview** | Stage badge grid, metrics, summary table |
| **Learning Curves** | Multi-stage win-rate trajectories + per-stage zoom |
| **Strategy Evolution** | Parallel coordinates, per-param time series, radar |
| **Agent Inspector** | Parameter bar chart + generation log table |
| **OOD Dashboard** | AUROC/TPR/FPR bars + scenario flagging rates |
| **Value Learning** | Weight convergence + learned vs. true comparison |

### Launch the interactive dashboard

```bash
pip install streamlit pandas matplotlib
streamlit run prometheus_dashboard.py
```

### Use real run data

```python
from prometheus.strategy_data import save_run
# after a real OpenRA or MicroRTS curriculum run:
save_run(stage_results, 'my_run.json')
# then enter 'my_run.json' in the dashboard sidebar
```